# IMTalker - Generator Training (notebook 4/4)

Trains the **Generator** (flow-matching transformer) on the dataset produced by
notebooks `01`-`03`, using the repository's own training implementation
(`generator/train.py`) and inference pipeline (`generator/generate.py`).

**What this notebook does**
- Uses the repo's `System` (FMGenerator + L1 flow/velocity losses + EMA), dataset,
  options and Trainer configuration - **no repo file is modified**.
- **Multi-GPU**: launches the repo's exact DDP setup
  (`strategy='ddp_find_unused_parameters_true'`, `devices=-1`) as a background
  process (Option A). Single GPU can also train directly in the notebook (Option B).
- **Logs**: TensorBoard + `metrics.csv` + human-readable `train_log.txt`
  (epoch / step / loss / lr).
- **Checkpoints**: saved every `SAVE_FREQ` steps + `last.ckpt`; training
  **auto-resumes** from `last.ckpt` after an interruption.
- After **every epoch**, renders a test video with the current weights using the
  repo's `InferenceAgent` -> `exps/<exp_name>/test_videos/epoch_XXXX.mp4`.

**Prerequisites**
1. Notebooks `01`-`03` completed -> `preprocessing_output/generator_dataset/{motion,audio,smirk,gaze}`
2. `preprocessing_output/checkpoints/renderer.ckpt` + `wav2vec2-base-960h/` (from notebooks 01/02)
3. A test reference image + a test `.wav` (defaults auto-pick from `assets/`)

**Note (repo bug workaround, no repo files changed):** `generator/dataset.py`
returns keys `gaze_now/pose_now/cam_now` while `generator/FM.py` reads
`gaze/pose/cam`; the wrapper script renames them on the fly (the repo's own
`train.py` would otherwise raise `KeyError` on the first batch).

In [ ]:
# ============================================================================
# CONFIGURATION - EDIT THE PATHS / SETTINGS BELOW
# ============================================================================
from pathlib import Path
import torch

# --- Paths -------------------------------------------------------------------
REPO_ROOT = Path(r"/workspace/IMTalker")                     # repo root on the Pod
DATASET_DIR = REPO_ROOT / "preprocessing_output" / "generator_dataset"   # from notebooks 01-03
CKPT_DIR = REPO_ROOT / "preprocessing_output" / "checkpoints"            # from notebooks 01/02
WAV2VEC_DIR = CKPT_DIR / "wav2vec2-base-960h"
RENDERER_CKPT = CKPT_DIR / "renderer.ckpt"

EXP_ROOT = REPO_ROOT / "exps"          # training outputs (logs, checkpoints, test videos)
EXP_NAME = "generator_exp"

# --- Training hyper-parameters (repo defaults in comments) -------------------
BATCH_SIZE = 16            # per GPU (repo default: 16)
LR = 1e-4                  # repo default: 1e-4
MAX_STEPS = 200_000        # repo default: 5_000_000 - lower this for small datasets
MAX_EPOCHS = -1            # -1 = unlimited (stop at MAX_STEPS); set e.g. 50 to stop by epochs
SAVE_FREQ = 5_000          # checkpoint every N steps (repo default: 100_000)
DISPLAY_FREQ = 10_000      # validation + metric logging interval in steps (repo default: 10_000)
LOG_EVERY_N_STEPS = 50     # train_log.txt write interval

AUTO_RESUME = True         # resume from checkpoints/last.ckpt if present
RESUME_CKPT = None         # or set an explicit checkpoint path to resume from

# --- Per-epoch test video -----------------------------------------------------
TEST_REF_IMAGE = None      # None = auto-pick (assets/source_3.jpg, else a frame from videos_512)
TEST_AUDIO = None          # None = auto-pick (assets/audio_1.wav)
TEST_EVERY_N_EPOCHS = 1    # render a test video after every N epochs
TEST_CROP = True           # auto-crop the face in the test image (repo --crop)
TEST_CFG_SCALE = 3.0       # repo default a_cfg_scale
TEST_NFE = 10              # repo default ODE steps
TEST_SEED = 42

# --- Wrapper files (created alongside this notebook) --------------------------
SCRIPTS_DIR = REPO_ROOT / "preprocessing"
TRAIN_SCRIPT = SCRIPTS_DIR / "run_generator_training.py"
UTILS_MODULE = SCRIPTS_DIR / "imtalker_train_utils.py"

# --- Validation ----------------------------------------------------------------
assert REPO_ROOT.exists(), f"REPO_ROOT not found: {REPO_ROOT} - edit the config above!"
assert TRAIN_SCRIPT.exists(), f"missing {TRAIN_SCRIPT} (should sit next to this notebook)"
assert UTILS_MODULE.exists(), f"missing {UTILS_MODULE} (should sit next to this notebook)"
EXP_DIR = EXP_ROOT / EXP_NAME
print(f"repo:    {REPO_ROOT}")
print(f"dataset: {DATASET_DIR}")
print(f"outputs: {EXP_DIR}")

In [ ]:
# Environment & dependency check (installs only what is missing).
import importlib.util, shutil, subprocess, sys

def _pip_install(pkgs):
    print("[install]", " ".join(pkgs))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

required = {
    "pytorch_lightning": "pytorch-lightning==2.2.1",   # repo pin (requirement.txt)
    "einops": "einops",
    "torchdiffeq": "torchdiffeq==0.2.5",               # repo pin
    "transformers": "transformers>=4.38,<4.42",        # same as preprocessing notebooks
    "librosa": "librosa",
    "face_alignment": "face_alignment==1.4.1",         # repo pin
    "av": "av==12.0.0",                                # repo pin (video writing)
    "tensorboard": "tensorboard",
    "timm": "timm==1.0.9",                             # repo pin
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    _pip_install(missing)
else:
    print("all required packages already installed")

import torch, pytorch_lightning as pl
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    print(f"  GPU[{i}] {torch.cuda.get_device_name(i)}")
print("pytorch-lightning:", pl.__version__, "" if pl.__version__ == "2.2.1" else "(repo pins 2.2.1)")
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found - generator training requires a GPU.")
assert shutil.which("ffmpeg"), "ffmpeg not found - needed to mux audio into the test videos"
print("ffmpeg:", shutil.which("ffmpeg"))

In [ ]:
# Pre-flight checks: dataset, checkpoints, test files.
import cv2

# 1) dataset produced by notebooks 01-03
for sub in ("motion", "audio", "smirk", "gaze"):
    d = DATASET_DIR / sub
    assert d.exists(), f"missing {d} - run notebooks 01-03 first"
counts = {s: len(list((DATASET_DIR / s).glob("*.*"))) for s in ("motion", "audio", "smirk", "gaze")}
print("dataset files:", counts)
assert counts["motion"] >= 2, "need at least 2 preprocessed clips"

# 2) wav2vec2 (downloaded by notebook 02)
assert WAV2VEC_DIR.exists() and any(WAV2VEC_DIR.iterdir()), \
    f"missing {WAV2VEC_DIR} - run notebook 02 (or download wav2vec2-base-960h)"

# 3) renderer.ckpt (downloaded by notebook 01; download here if missing)
if not RENDERER_CKPT.exists():
    from huggingface_hub import hf_hub_download
    print("[download] renderer.ckpt (~2.1 GB) from cbsjtu01/IMTalker ...")
    hf_hub_download(repo_id="cbsjtu01/IMTalker", filename="renderer.ckpt", local_dir=str(CKPT_DIR))
assert RENDERER_CKPT.exists(), f"missing {RENDERER_CKPT}"

# 4) test reference image + audio (auto-pick if not set in CONFIG)
if TEST_REF_IMAGE is None:
    cand = REPO_ROOT / "assets" / "source_3.jpg"
    if cand.exists():
        TEST_REF_IMAGE = cand
    else:
        vids = sorted((REPO_ROOT / "preprocessing_output" / "videos_512").glob("*.mp4"))
        assert vids, "set TEST_REF_IMAGE in the CONFIG cell (no assets/source_3.jpg and no videos_512)"
        cap = cv2.VideoCapture(str(vids[0]))
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2)
        ok, frame = cap.read(); cap.release()
        assert ok, f"could not read a frame from {vids[0]}"
        TEST_REF_IMAGE = EXP_DIR / "test_ref.png"
        TEST_REF_IMAGE.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(TEST_REF_IMAGE), frame)
if TEST_AUDIO is None:
    cand = REPO_ROOT / "assets" / "audio_1.wav"
    assert cand.exists(), "set TEST_AUDIO in the CONFIG cell (assets/audio_1.wav not found)"
    TEST_AUDIO = cand
TEST_REF_IMAGE, TEST_AUDIO = Path(TEST_REF_IMAGE), Path(TEST_AUDIO)
assert TEST_REF_IMAGE.exists() and TEST_AUDIO.exists()

n_gpu = torch.cuda.device_count()
strategy = "ddp_find_unused_parameters_true" if n_gpu > 1 else "auto"
print(f"test image: {TEST_REF_IMAGE}")
print(f"test audio: {TEST_AUDIO}")
print(f"gpus={n_gpu} -> strategy={strategy} | effective batch = {BATCH_SIZE} x {n_gpu} = {BATCH_SIZE * n_gpu}")

## Launch training

**Option A (recommended; required for multi-GPU)** - launches the training as a
background process with the repository's exact DDP configuration. It keeps
running even if you close this notebook / restart the kernel, and writes
`train_console.log` + `train_log.txt`.

**Option B (single GPU only)** - runs the same training code directly in this
notebook (interactive progress bar; the cell blocks until training finishes).

Both options use the identical wrapper script (`preprocessing/run_generator_training.py`),
which reuses the repo's `System`, dataset, options and Trainer configuration.

In [ ]:
# OPTION A - launch training as a background process (recommended; required for multi-GPU).
# Uses the repository's exact DDP configuration. Survives notebook/kernel restarts.
import os, subprocess, sys

console_log = EXP_DIR / "train_console.log"
EXP_DIR.mkdir(parents=True, exist_ok=True)

cmd = [sys.executable, str(TRAIN_SCRIPT),
    "--repo_root", str(REPO_ROOT),
    "--dataset_path", str(DATASET_DIR),
    "--exp_root", str(EXP_ROOT),
    "--exp_name", EXP_NAME,
    "--wav2vec_model_path", str(WAV2VEC_DIR),
    "--renderer_ckpt", str(RENDERER_CKPT),
    "--batch_size", str(BATCH_SIZE),
    "--lr", str(LR),
    "--max_steps", str(MAX_STEPS),
    "--max_epochs", str(MAX_EPOCHS),
    "--save_freq", str(SAVE_FREQ),
    "--display_freq", str(DISPLAY_FREQ),
    "--log_every_n_steps", str(LOG_EVERY_N_STEPS),
    "--test_ref", str(TEST_REF_IMAGE),
    "--test_aud", str(TEST_AUDIO),
    "--test_out_dir", str(EXP_DIR / "test_videos"),
    "--test_every_n_epochs", str(TEST_EVERY_N_EPOCHS),
    "--test_crop", "1" if TEST_CROP else "0",
    "--test_cfg", str(TEST_CFG_SCALE),
    "--test_nfe", str(TEST_NFE),
    "--test_seed", str(TEST_SEED),
]
if not AUTO_RESUME:
    cmd.append("--no_auto_resume")
if RESUME_CKPT:
    cmd += ["--resume_ckpt", str(RESUME_CKPT)]

print("launching:", " ".join(cmd), "\n")
with open(console_log, "a") as logf:
    popen_kwargs = dict(stdout=logf, stderr=subprocess.STDOUT, cwd=str(REPO_ROOT))
    if os.name == "posix":
        popen_kwargs["start_new_session"] = True   # detach: survives kernel restarts
    else:
        popen_kwargs["creationflags"] = subprocess.CREATE_NEW_PROCESS_GROUP
    proc = subprocess.Popen(cmd, **popen_kwargs)

(EXP_DIR / "train.pid").write_text(str(proc.pid))
print(f"training started | pid={proc.pid}")
print(f"console log: {console_log}")
print("run the MONITOR cell below to follow progress.")

In [ ]:
# OPTION B - train directly in this notebook (single GPU only; blocks this cell).
# For multi-GPU use OPTION A (the repo's DDP must be launched as a script).
assert torch.cuda.device_count() == 1, \
    f"{torch.cuda.device_count()} GPUs detected - use OPTION A (background launch) for multi-GPU."

import argparse, sys
for p in (str(SCRIPTS_DIR), str(REPO_ROOT), str(REPO_ROOT / "generator")):
    if p not in sys.path:
        sys.path.insert(0, p)

import run_generator_training as rt

args = argparse.Namespace(
    repo_root=str(REPO_ROOT), dataset_path=str(DATASET_DIR),
    exp_root=str(EXP_ROOT), exp_name=EXP_NAME,
    wav2vec_model_path=str(WAV2VEC_DIR), renderer_ckpt=str(RENDERER_CKPT),
    batch_size=BATCH_SIZE, lr=LR, max_steps=MAX_STEPS, max_epochs=MAX_EPOCHS,
    save_freq=SAVE_FREQ, display_freq=DISPLAY_FREQ,
    resume_ckpt=str(RESUME_CKPT) if RESUME_CKPT else None,
    no_auto_resume=not AUTO_RESUME,
    test_ref=str(TEST_REF_IMAGE), test_aud=str(TEST_AUDIO),
    test_out_dir=str(EXP_DIR / "test_videos"),
    test_every_n_epochs=TEST_EVERY_N_EPOCHS, test_crop=1 if TEST_CROP else 0,
    test_cfg=TEST_CFG_SCALE, test_nfe=TEST_NFE, test_seed=TEST_SEED,
    log_every_n_steps=LOG_EVERY_N_STEPS,
)
rt.main(args)

In [ ]:
# MONITOR - re-run this cell anytime to check progress.
import os
from pathlib import Path

def _tail(path, n):
    path = Path(path)
    if not path.exists():
        print(f"(no {path.name} yet)")
        return
    lines = path.read_text(errors="replace").splitlines()
    print(f"----- last {min(n, len(lines))} lines of {path.name} -----")
    print("\n".join(lines[-n:]))

# process status (Option A)
pid_file = EXP_DIR / "train.pid"
if pid_file.exists():
    pid = int(pid_file.read_text().strip())
    try:
        import psutil
        alive = psutil.pid_exists(pid)
    except ImportError:
        try:
            os.kill(pid, 0); alive = True
        except OSError:
            alive = False
    print(f"training process pid={pid}: {'RUNNING' if alive else 'not running'}")

_tail(EXP_DIR / "train_log.txt", 20)

# metrics.csv (CSVLogger)
csvs = sorted(EXP_DIR.glob("version_*/metrics.csv"))
if csvs:
    _tail(csvs[-1], 5)

# checkpoints
cks = sorted((EXP_DIR / "checkpoints").glob("*.ckpt"), key=lambda p: p.stat().st_mtime)
print(f"\ncheckpoints ({len(cks)}):", [c.name for c in cks[-4:]])

# per-epoch test videos
tvs = sorted((EXP_DIR / "test_videos").glob("epoch_*.mp4"))
print(f"test videos ({len(tvs)}):", [t.name for t in tvs[-4:]])
if tvs:
    print("latest:", tvs[-1])

In [ ]:
# STOP training started with OPTION A (graceful SIGTERM; checkpoints up to the
# last SAVE_FREQ are kept and AUTO_RESUME will continue from last.ckpt).
STOP_TRAINING = False   # set to True and re-run this cell to actually stop

if not STOP_TRAINING:
    print("set STOP_TRAINING = True and re-run this cell to stop training.")
else:
    import os, signal, subprocess
    pid_file = EXP_DIR / "train.pid"
    assert pid_file.exists(), "no train.pid - was training launched with OPTION A?"
    pid = int(pid_file.read_text().strip())
    if os.name == "posix":
        os.kill(pid, signal.SIGTERM)
    else:
        subprocess.run(["taskkill", "/PID", str(pid), "/T", "/F"])
    print(f"stop signal sent to pid {pid}")

## After training

**Outputs** (in `exps/<EXP_NAME>/`):
- `checkpoints/` - `step=XXXXXX.ckpt` every `SAVE_FREQ` steps + `last.ckpt` (auto-resume source)
- `train_log.txt` - human-readable log (epoch / step / loss / lr)
- `train_console.log` - full console output (Option A)
- `version_*/` - TensorBoard events + `metrics.csv`
- `test_videos/epoch_XXXX.mp4` - test video rendered after every epoch

**Resume:** just re-launch (Option A or B) - `AUTO_RESUME=True` picks up `last.ckpt`
with the full training state (step, optimizer, scheduler, EMA).

**Inference with your trained generator** (repo command):
```bash
cd <REPO_ROOT>/generator
python generate.py \
    --ref_path assets/source_3.jpg \
    --aud_path assets/audio_1.wav \
    --res_video_path results/my_video.mp4 \
    --generator_path <REPO_ROOT>/exps/<EXP_NAME>/checkpoints/last.ckpt \
    --renderer_path <REPO_ROOT>/preprocessing_output/checkpoints/renderer.ckpt \
    --wav2vec_model_path <REPO_ROOT>/preprocessing_output/checkpoints/wav2vec2-base-960h \
    --crop
```